In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%matplotlib inline

In [2]:
words = open('names.txt', 'r').read().splitlines()
print(words[:8])
len(words)

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']


32033

In [3]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)
print(itos)
print(vocab_size)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}
27


In [4]:
block_size = 3 # context length

def build_dataset(words):  
  X, Y = [], []
  
  for w in words:
    context = [0] * block_size
    for ch in w + '.':
      ix = stoi[ch]
      X.append(context)
      Y.append(ix)
      context = context[1:] + [ix] # crop and append

  X = torch.tensor(X)
  Y = torch.tensor(Y)
  print(X.shape, Y.shape)
  return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr,  Ytr  = build_dataset(words[:n1])     
Xdev, Ydev = build_dataset(words[n1:n2])   
Xte,  Yte  = build_dataset(words[n2:])     

torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


In [5]:
# utility function we will use later when comparing manual gradients to PyTorch gradients
def cmp(s, dt, t):
  ex = torch.all(dt == t.grad).item()
  app = torch.allclose(dt, t.grad)
  maxdiff = (dt - t.grad).abs().max().item()
  print(f'{s:15s} | exact: {str(ex):5s} | approximate: {str(app):5s} | maxdiff: {maxdiff}')

In [6]:
n_embd = 10 # the dimensionality of the character embedding vectors
n_hidden = 64 # the number of neurons in the hidden layer of the MLP

g = torch.Generator().manual_seed(2147483647) # for reproducibility
C  = torch.randn((vocab_size, n_embd),            generator=g)
# Layer 1
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g) * (5/3)/((n_embd * block_size)**0.5)
b1 = torch.randn(n_hidden,                        generator=g) * 0.1 # using b1 just for fun, it's useless because of BN
# Layer 2
W2 = torch.randn((n_hidden, vocab_size),          generator=g) * 0.1
b2 = torch.randn(vocab_size,                      generator=g) * 0.1
# BatchNorm parameters
bngain = torch.randn((1, n_hidden))*0.1 + 1.0
bnbias = torch.randn((1, n_hidden))*0.1

# initializating many of these parameters in non-standard ways
# because sometimes initializating with e.g. all zeros could mask an incorrect
# implementation of the backward pass.

parameters = [C, W1, b1, W2, b2, bngain, bnbias]
print(sum(p.nelement() for p in parameters)) # number of parameters in total
for p in parameters:
  p.requires_grad = True

4137


In [7]:
batch_size = 32
n = batch_size # a shorter variable also, for convenience
# construct a minibatch
ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
Xb, Yb = Xtr[ix], Ytr[ix] # batch X,Y

In [8]:
# forward pass, "chunkated" into smaller steps that are possible to backward one at a time

emb = C[Xb] # embed the characters into vectors
embcat = emb.view(emb.shape[0], -1) # concatenate the vectors
# Linear layer 1
hprebn = embcat @ W1 + b1 # hidden layer pre-activation
# BatchNorm layer
bnmeani = 1/n*hprebn.sum(0, keepdim=True)
bndiff = hprebn - bnmeani
bndiff2 = bndiff**2
bnvar = 1/(n-1)*(bndiff2).sum(0, keepdim=True) # note: Bessel's correction (dividing by n-1, not n)
bnvar_inv = (bnvar + 1e-5)**-0.5
bnraw = bndiff * bnvar_inv
hpreact = bngain * bnraw + bnbias
# Non-linearity
h = torch.tanh(hpreact) # hidden layer
# Linear layer 2
logits = h @ W2 + b2 # output layer
# cross entropy loss (same as F.cross_entropy(logits, Yb))
logit_maxes = logits.max(1, keepdim=True).values
norm_logits = logits - logit_maxes # subtract max for numerical stability
counts = norm_logits.exp()
counts_sum = counts.sum(1, keepdims=True)
counts_sum_inv = counts_sum**-1 # if I use (1.0 / counts_sum) instead then I can't get backprop to be bit exact...
probs = counts * counts_sum_inv
logprobs = probs.log()
loss = -logprobs[range(n), Yb].mean()

# PyTorch backward pass
for p in parameters:
  p.grad = None
for t in [logprobs, probs, counts, counts_sum, counts_sum_inv, # afaik there is no cleaner way
          norm_logits, logit_maxes, logits, h, hpreact, bnraw,
         bnvar_inv, bnvar, bndiff2, bndiff, hprebn, bnmeani,
         embcat, emb]:
  t.retain_grad()
loss.backward()
loss

tensor(3.3483, grad_fn=<NegBackward0>)

In [9]:
bnvar.shape , bndiff2.shape

(torch.Size([1, 64]), torch.Size([32, 64]))

In [22]:
# loss = -(a+b+c)/3
# dloss/da = -1/3 in general its - 1/n
dlogprobs= torch.zeros_like(logprobs)
dlogprobs[range(n), Yb]=-1.0/n

dprobs= (1.0/probs) * dlogprobs # local derivative * chain rule

# # c= a*b
# # a =
# [a11 a12 a13
#  a21 a22 a23
#  a31 a32 a33]

# b =
# [b1
#  b2
#  b3]

# Broadcast gives

# [a11*b1  a12*b1  a13*b1
#  a21*b2  a22*b2  a23*b2
#  a31*b3  a32*b3  a33*b3] if a node is used multiple times their gradients should summed during backpropagation as seen in micrograd
#Whenever one tensor is used in multiple places in the forward pass, its gradient is the sum of the gradients coming from every path.
dcounts_sum_inv= (dprobs*counts).sum( 1,keepdim=True) # retain dimension so that count_sum_inv and its gradient are the same shape

dcounts_sum=(-counts_sum**-2)*dcounts_sum_inv

# a11 a12 a13 ---> b1 (=a11+a12+a13)
# derivative of b1 will flow into all a11, a12, a13 equally

dcounts= counts_sum_inv*dprobs
dcounts+= torch.ones_like(counts)*dcounts_sum

dnorm_logits=dcounts*counts#(norm_logits).exp() # d/dx(e^x)=e^x

#norm_logits.shape, logits.shape,logit_maxes.shape
# c11 c12 c13 = a11 a12 a13 -b1 
# eg c 12 = a12 -b1 derivative 1 wrt a and -1 wrt b but need to sum like before due to shape issues
dlogits=dnorm_logits.clone() # it also has another component
dlogit_maxes= -dlogits.sum(1, keepdim=True)
# should be extremely small because it doesnt affect probs one could assume its exactly zero but with floating point arithmetic its extremely small but not zero

#dlogits += F.one_hot(logits.max(1).indices, num_classes=logits.shape[1]) * dlogit_maxes ----> shows where the max came from and since its one hot it makes it one and then multiply by incoming derivative(chain rule)
dlogits += F.one_hot(logits.max(1).indices, num_classes=logits.shape[1]) * dlogit_maxes

# dlogits.shape, h.shape, W2.shape, b2.shape # look at notebook for derivation
# dl/da= dl/dd @ bt 
# dl/dc = dl/dd.sum(0) along columns
dh= dlogits @ W2.T # easier way is dh has to be the same shape as h which is 32,64 its some matrix mul of logits and something logirs is 32,27 and W2 is 64,27 so logically too h has to be logits @ W2.T
dW2=h.T @ dlogits 
db2= dlogits.sum(0) # have to remove the first dimension to keep it 27 

dhpreact= (1.0-h**2)*dh # seen in micrograd

dbngain= (bnraw*dhpreact).sum(0, keepdim =True)
dbnraw= (bngain *dhpreact) # maintain the shapes and you will know
dbnbias=dhpreact.sum(0,keepdim=True)

dbndiff= bnvar_inv*dbnraw # not done yet
dbnvar_inv= (bndiff*dbnraw).sum(0,keepdim=True) # just like last time

dbnvar= (-0.5* (bnvar + 1e-5)**-1.5 *1.0)*dbnvar_inv

# whenever there is a sum in forward pass it becomes a replication in backwards pass vice versa as that indicates variable reuse
# a11 a12
# a21 a22 ---> b1 b2 where
# b1= 1/(n-1)(a11+a12)
# b2= 1/(n-1)(a21+a22)

dbndiff2= (1.0/(n-1))*torch.ones_like(bndiff2)*dbnvar

dbndiff+=(2*bndiff)*dbndiff2

dhprebn=dbndiff.clone() # another factor remaining

dbnmeani= -dbndiff.sum(0)

dhprebn+=torch.ones_like(hprebn)*(1.0/n)*dbnmeani # broadcasting again

dembcat= dhprebn @ W1.T
dW1= embcat.T @ dhprebn
db1= dhprebn.sum(0)

demb=dembcat.view(emb.shape) # just change the shape deconcatenate

dC=torch.zeros_like(C)
for i in range(Xb.shape[0]):
    for j in range(Xb.shape[1]):
        ix=Xb[i,j]
        dC[ix]+=demb[i,j]


In [23]:

cmp('logprobs', dlogprobs, logprobs)
cmp('probs', dprobs, probs)
cmp('counts_sum_inv', dcounts_sum_inv, counts_sum_inv)
cmp('counts_sum', dcounts_sum, counts_sum)
cmp('counts', dcounts, counts)
cmp('norm_logits', dnorm_logits, norm_logits) 
cmp('logit_maxes', dlogit_maxes, logit_maxes)
cmp('logits', dlogits, logits)
cmp('h', dh, h)
cmp('W2', dW2, W2)
cmp('b2', db2, b2)
cmp('hpreact', dhpreact, hpreact)
cmp('bngain', dbngain, bngain)
cmp('bnbias', dbnbias, bnbias)
cmp('bnraw', dbnraw, bnraw)
cmp('bnvar_inv', dbnvar_inv, bnvar_inv)
cmp('bnvar', dbnvar, bnvar)
cmp('bndiff2', dbndiff2, bndiff2)
cmp('bndiff', dbndiff, bndiff)
cmp('bnmeani', dbnmeani, bnmeani)
cmp('hprebn', dhprebn, hprebn)
cmp('embcat', dembcat, embcat)
cmp('W1', dW1, W1)
cmp('b1', db1, b1)
cmp('emb', demb, emb)
cmp('C', dC, C)


logprobs        | exact: True  | approximate: True  | maxdiff: 0.0
probs           | exact: True  | approximate: True  | maxdiff: 0.0
counts_sum_inv  | exact: True  | approximate: True  | maxdiff: 0.0
counts_sum      | exact: True  | approximate: True  | maxdiff: 0.0
counts          | exact: True  | approximate: True  | maxdiff: 0.0
norm_logits     | exact: True  | approximate: True  | maxdiff: 0.0
logit_maxes     | exact: True  | approximate: True  | maxdiff: 0.0
logits          | exact: True  | approximate: True  | maxdiff: 0.0
h               | exact: True  | approximate: True  | maxdiff: 0.0
W2              | exact: True  | approximate: True  | maxdiff: 0.0
b2              | exact: True  | approximate: True  | maxdiff: 0.0
hpreact         | exact: True  | approximate: True  | maxdiff: 0.0
bngain          | exact: True  | approximate: True  | maxdiff: 0.0
bnbias          | exact: True  | approximate: True  | maxdiff: 0.0
bnraw           | exact: True  | approximate: True  | maxdiff:

In [11]:
# implementing the second part of logits as done with lognorms
# need help with this gradient
# logit_maxes = logits.max(1, keepdim=True).values
# max_indices = logits.max(1, keepdim=True).indices
# dlogits[torch.arange(n), max_indices.squeeze()] += dlogit_maxes.squeeze()